[back to [index.ipynb](index.ipynb)]

## Building the Michelson entity data set

- Read the raw 1879 instrument readings from `data/michelson-1879.dat`
- attach the units they were recorded in
- converts to SI
- wraps each column in an `Entity` carrying its EMMO term and description
- compute displacement $d$
- write the resulting `EntityCollection` as CSV, YAML and HDF5.
  - the content of the file is the same as `data/michelson-entities-1879.csv`



In [ ]:
import mammos_entity as me
import mammos_units as u
import numpy as np
from astropy.units import imperial  # special case: needed to get to feet as unit 

datafile = "data/michelson-1879.dat"

### Reading the raw data

`data/michelson-1879.dat` is headerless and unitless: 100 observations, four columns.

| column | meaning | unit as recorded |
|---|---|---|
| 1 | rotation rate of the revolving mirror | revolutions per second |
| 2 | revolving mirror to the crosshair of the micrometer | feet |
| 3 | displacement of the returned image | turns of the micrometer screw |
| 4 | travel of the crosshair per turn of that screw | millimetres per turn |

One number the apparatus needed is not in the table at all: the separation of the
revolving and the fixed mirror, `D = 1986.23 ft`.

In [ ]:
# reed all 100 lines of data with 4 columns
raw = np.loadtxt(datafile)
raw[0:5, :]

In [ ]:
raw.shape  # shows how many (rows, columns) of data we have

### 2. Quantities, in the units the instrument used

In [ ]:
# Extract arrays of data from file (metadata comes from paper)

N     = raw[:, 0] / u.s          # revolutions per second
r     = raw[:, 1] * imperial.ft  # radial distance in feet
turns = raw[:, 2]                # turns of the screw (dimensionless)
pitch = raw[:, 3] * u.mm         # millimetres per turn of screw

D = 1986.23 * imperial.ft        # mirror separation 

print(f"D     = {D:.4f}")        # print some example values
print(f"r[0]  = {r[0]:.4f}")

### 3. Convert quantities to SI

In [ ]:
N_si     = N.to(u.Hz)   # rotations per second in Hertz
r_si     = r.to(u.m)    # lengths in metre
pitch_si = pitch.to(u.m)
D_si     = D.to(u.m)

print(f"D     = {D_si:.4f}")    # print some example values
print(f"r[0]  = {r_si[0]:.4f}")


`turns` is dimensionless, no conversion needed.

### 4. Create entities from Quantities

In [ ]:
N = me.Entity(
    "Frequency",
    N_si,
    description=(
        "Rotation rate of the revolving mirror, in revolutions per second, "
        "timed against a tuning fork. "
    ),
)

In [ ]:
N

In [ ]:
r = me.Entity(
    "RadialDistance",
    r_si,
    description=(
        "Distance from the axis of the revolving mirror to the crosshair of the "
        "micrometer."
    ),
)

In [ ]:
screw_turns = me.Entity(
    "AngularDisplacement",
    turns,
    description=(
        "Rotation of the micrometer screw between the setting on the slit and the "
        "setting on the deflected image. Measured in in turns of the "
        "screw, not necessarily a whole number. Dimensionless."
    ),
)

In [ ]:
screw_pitch = me.Entity(
    "Length",
    pitch_si,
    description=(
        "Travel of the micrometer crosshair per one full turn of the screw. "
        "Recorded in millimetres per turn. The pitch varies along the screw, hence one calibration value per observation. "
        "EMMO has no term for screw pitch; Length is the closest I could find."
    ),
)

## 5. Compute distance $d$ (derived entities) 

In [ ]:
d_si = pitch_si * turns
d_si[0]

In [ ]:
d = me.Entity(
    "Distance",
    d_si,
    description=(
        "Separation of the slit and the deflected image, obtained as "
        "screw_turns * screw_pitch. "
    ),
)

d

## 6. Entity Collection (the whole data set)

Create a description:

In [ ]:
description = f"""Michelson's 1879 determination of the speed of light: the instrument readings.

A. A. Michelson, "Experimental Determination of the Velocity of Light", Astronomical Papers of the American Ephemeris 1 (1880) 109-145. 100 observations made at the U.S. Naval Academy, Annapolis, 5 June - 2 July 1879. Transcribed via the R package loon.data (michelson_1879); see also MacKay and Oldford, Statistical Science 15(3) 254-278 (2000), doi:10.1214/ss/1009212817.

Rotating-mirror method. Light travels from the revolving mirror to a fixed mirror and back while the mirror turns; the returned beam is deflected by twice the mirror rotation and read at radial distance r as a displacement d, giving c = 8 * pi * N * D * r / d

The apparatus constant D is the separation of the revolving and the fixed mirror. It is not part of the observation table and is recorded only here: D = 1986.23 ft = {D_si.value:.4f} m

The revolving mirror rotates with frequence N. The speed of light is not stored in this file: it is derived from these readings. Note that the light travelled in air, not in vacuum."""

Turn all entities together into a *entity collection* to represent the data from the experiment.

In [ ]:
experiment = me.EntityCollection(
    description,
    N=N,
    r=r,
    screw_turns=screw_turns,
    screw_pitch=screw_pitch,
    d=d,
)
experiment

## 7. Write data from experiment to disk

The entity collection in `data/michelson-entities-1879.csv` has been created using the above code, followed by `experiment.to_csv("data/michelson-entities-1879.csv")`.

We save the experiment here to different files so they are easier to inspect if desired.

In [ ]:
experiment.to_csv("ec.csv")
experiment.to_yaml("ec.yaml")
experiment.to_hdf5("ec.h5")